# Exercises — Strategy return analysis

[DataCamp exercise](https://campus.datacamp.com/courses/financial-trading-in-python/performance-evaluation-4?ex=1) · see `Notes.md` in this folder for the summary.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project's data folder regardless of where this notebook runs from
DATA = next(p / "course materials" / "data"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course materials" / "data").is_dir())

def load(name):
    """Load an OHLCV CSV with a parsed DatetimeIndex."""
    return pd.read_csv(DATA / name, index_col="Date", parse_dates=True)

def price(name, col, year=None):
    """Single-asset price DataFrame (column = `col`) for use with bt."""
    df = load(name)
    if year:
        df = df[df.index.year == year]
    return df["Close"].rename(col).to_frame()


In [ ]:
import bt
import talib

data = price("AMZN-stock-data.csv", "AMZN")          # full range
sma = talib.SMA(data["AMZN"], timeperiod=50)
signal = pd.DataFrame(data["AMZN"].values > sma.values, index=data.index, columns=["AMZN"])
strat = bt.Strategy("SMA50", [bt.algos.SelectWhere(signal),
                              bt.algos.WeighEqually(), bt.algos.Rebalance()])
bt_result = bt.run(bt.Backtest(strat, data))

# Return statistics
resInfo = bt_result.stats
print(resInfo.loc[["daily_mean", "monthly_mean", "yearly_mean", "cagr"]])

### Return histogram

In [ ]:
bt_result.plot_histograms(bins=40, freq="w")
plt.show()